In [ ]:
#python -m pip install --upgrade pip
#pip install transformers==4.46.3 accelerate bitsandbytes peft trl datasets huggingface_hub wandb scipy

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer,BitsAndBytesConfig
model_id = 'Qwen/Qwen2.5-0.5B-Instruct'
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16    
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map='auto')

In [ ]:
from datasets import load_dataset
# 범용 비즈니스 및 지시어 데이터셋 (KoAlpaca) 로드
dataset = load_dataset("beomi/KoAlpaca-v1.1a", split="train")
# 모델이 알아들을 수 있도록 Qwen Chat Template 적용 함수
def format_prompt(example):
    messages = [
        {"role": "system", "content": "You are a helpful AI assistant. Respond in Korean."},
        {"role": "user", "content": example["instruction"]}
    ]
    
    # 1. 챗 템플릿 적용
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    # 2.  만약 버그로 인해 리스트(숫자)가 반환되었다면 강제로 문자열로 디코딩
    if isinstance(text, list):
        text = tokenizer.decode(text)
        
    # 3. 정답(output)과 종료 토큰 추가
    text += example["output"] + tokenizer.eos_token
    return {"text": text}
# 실습 시간 단축을 위해 2,000건의 데이터만 샘플링하여 훈련 세트로 구성
train_dataset = dataset.select(range(2000)).map(format_prompt)
print("\n[전처리된 데이터 예시]")
print(train_dataset[0]["text"])

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# k-bit 양자화된 모델을 파인튜닝할 수 있도록 전처리
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 최신 trl 라이브러리에 맞추어 TrainingArguments 대신 SFTConfig 사용
training_args = SFTConfig(
    output_dir="./qwen-koalpaca-lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=100, # 100 step만 진행
    optim="paged_adamw_8bit",
    fp16=False,
    bf16=True, # H100의 BFloat16 가속 활성화
    dataset_text_field="text", 
    max_seq_length=512,        
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args,
    peft_config=lora_config
)

In [ ]:
# 모델 파인튜닝
trainer.train()